# Adalat Channel Cache Preparation (CPU)

This notebook performs the FFmpeg-heavy preparation for Notebook 14 without consuming GPU time. It builds the exact same deterministic train/evaluation view set, converts the selected GramVaani clips into mono 16 kHz WAVs, and saves resumable archive chunks to Drive.

Each completed chunk is durable. If Colab disconnects, open this notebook again on a CPU runtime and run all cells; completed chunks are skipped. No model is loaded and no training happens here.


## Before Running

1. Select **Runtime > Change runtime type > Hardware accelerator: None**.
2. Confirm the GramVaani archive and inventory from Notebook 10 remain in Drive.
3. Run all cells. Re-run after a cutoff until the final cell prints `CACHE PREPARATION COMPLETE`.
4. Then open `14_adalat_channel_adaptation_colab.ipynb`, select a T4, and run all cells.

Persistent cache chunks are saved under `MyDrive/call-whisper/results/channel_adaptation_adalat_small_seed0/serious/channel_cache_exact_frames_v1/`.


In [ ]:
# CPU-only setup. Fail immediately if a GPU runtime was selected.
import importlib
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')
nvidia_smi = shutil.which('nvidia-smi')
if nvidia_smi and subprocess.run([nvidia_smi], check=False, capture_output=True).returncode == 0:
    raise RuntimeError(
        'This is CPU-only preparation. Change Hardware accelerator to None, then run all.'
    )

REPO_DIR = Path('/content/CallWhisper-8k')
os.chdir('/content')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/anshulLuhsna/CallWhisper-8k.git', str(REPO_DIR),
], check=True)
os.chdir(REPO_DIR)
subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'libsndfile1'], check=True)

SRC_DIR = REPO_DIR / 'src'
sys.path.insert(0, str(SRC_DIR))
for module_name in list(sys.modules):
    if module_name == 'callwhisper' or module_name.startswith('callwhisper.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
channel_cache = importlib.import_module('callwhisper.datasets.channel_cache')
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Python:', platform.python_version())
print('Repository commit:', commit)
print('Runtime: CPU only')
print('Resumable cache module:', channel_cache.__file__)


In [ ]:
# Match Notebook 14's frozen profile and persistent paths.
import json

RUN_PROFILE = 'serious'  # smoke, pilot, serious, full
SEED = 0
EVAL_FRACTION = 0.05
PROFILES = {
    'smoke': {'max_train_sources': 256, 'max_eval_sources': 50, 'paired_eval_sources': 20},
    'pilot': {'max_train_sources': 4_000, 'max_eval_sources': 200, 'paired_eval_sources': 50},
    'serious': {'max_train_sources': 18_000, 'max_eval_sources': 500, 'paired_eval_sources': 100},
    'full': {'max_train_sources': None, 'max_eval_sources': 1_000, 'paired_eval_sources': 200},
}
PROFILE = PROFILES[RUN_PROFILE]
CHUNK_SIZE = 500
WORKERS = 8

DRIVE_PROJECT_DIR = Path('/content/drive/MyDrive/call-whisper')
INVENTORY_PATH = DRIVE_PROJECT_DIR / 'results/gv_train_100h_inventory/gv_train_100h_inventory.csv'
DATASET_ARCHIVE = DRIVE_PROJECT_DIR / 'saved_datasets/GV_Train_100h.tar.gz'
OUTPUT_DIR = DRIVE_PROJECT_DIR / 'results/channel_adaptation_adalat_small_seed0' / RUN_PROFILE
SPLIT_DIR = OUTPUT_DIR / 'splits'
PERSISTENT_CACHE_DIR = OUTPUT_DIR / 'channel_cache_exact_frames_v1'
WORK_ROOT = Path(f'/content/channel_cache_adalat_small_seed0_{RUN_PROFILE}')
LOCAL_DATA_ROOT = WORK_ROOT / 'dataset'
CACHE_BUILD_SCRATCH = WORK_ROOT / 'cache_build_scratch'
for directory in (OUTPUT_DIR, SPLIT_DIR, PERSISTENT_CACHE_DIR, WORK_ROOT, LOCAL_DATA_ROOT, CACHE_BUILD_SCRATCH):
    directory.mkdir(parents=True, exist_ok=True)
if not INVENTORY_PATH.exists():
    raise FileNotFoundError(f'Missing inventory: {INVENTORY_PATH}')
if not DATASET_ARCHIVE.exists():
    raise FileNotFoundError(f'Missing training archive: {DATASET_ARCHIVE}')
print(json.dumps({'run_profile': RUN_PROFILE, 'profile': PROFILE, 'chunk_size': CHUNK_SIZE}, indent=2))
print('Persistent cache:', PERSISTENT_CACHE_DIR)


In [ ]:
# Rebuild the exact deterministic split manifests used by Notebook 14.
from callwhisper.datasets.channel_adaptation import (
    build_artifacts, build_paired_eval_views, deterministic_limit, read_csv, write_csv,
)

split_summary = build_artifacts(
    INVENTORY_PATH, SPLIT_DIR, eval_fraction=EVAL_FRACTION, seed=SEED,
    max_train_sources=PROFILE['max_train_sources'],
    max_eval_sources=PROFILE['max_eval_sources'],
)
full_internal_eval = read_csv(SPLIT_DIR / 'internal_eval_source.csv')
paired_sources = deterministic_limit(
    full_internal_eval, PROFILE['paired_eval_sources'], seed=SEED + 2
)
write_csv(
    SPLIT_DIR / 'paired_internal_eval_views.csv', build_paired_eval_views(paired_sources)
)
print('Train views:', split_summary['train_view_rows'])
print('Internal evaluation views:', split_summary['internal_eval_view_rows'])


In [ ]:
# Copy and safely extract source audio to this CPU runtime's local disk.
import tarfile

LOCAL_ARCHIVE = WORK_ROOT / DATASET_ARCHIVE.name
EXTRACT_SENTINEL = LOCAL_DATA_ROOT / '.extract_complete'
if not LOCAL_ARCHIVE.exists() or LOCAL_ARCHIVE.stat().st_size != DATASET_ARCHIVE.stat().st_size:
    print('Copying 2.04 GB training archive from Drive...')
    shutil.copyfile(DATASET_ARCHIVE, LOCAL_ARCHIVE)
if not EXTRACT_SENTINEL.exists():
    print('Extracting training archive...')
    with tarfile.open(LOCAL_ARCHIVE, 'r:gz') as archive:
        archive.extractall(LOCAL_DATA_ROOT, filter='data')
    EXTRACT_SENTINEL.write_text('complete\n', encoding='utf-8')
dataset_candidates = [
    path for path in LOCAL_DATA_ROOT.rglob('GV_Train_100h') if (path / 'Audio').is_dir()
]
if not dataset_candidates and (LOCAL_DATA_ROOT / 'Audio').is_dir():
    dataset_candidates = [LOCAL_DATA_ROOT]
if not dataset_candidates:
    raise RuntimeError('Could not find extracted GV_Train_100h/Audio directory')
DATASET_DIR = dataset_candidates[0]
print('Dataset directory:', DATASET_DIR)


In [ ]:
# Build resumable cache archives. Completed chunks are skipped on every rerun.
from callwhisper.datasets.channel_cache import build_cache_chunks
from callwhisper.datasets.paired_telephony import validate_codec_support

codec_support = validate_codec_support()
if not all(codec_support.values()):
    raise RuntimeError(f'Colab ffmpeg lacks required codecs: {codec_support}')
train_views = read_csv(SPLIT_DIR / 'train_views.csv')
eval_views = read_csv(SPLIT_DIR / 'internal_eval_views.csv')
paired_views = read_csv(SPLIT_DIR / 'paired_internal_eval_views.csv')
all_views = train_views + eval_views + paired_views

def report_chunk(done: int, total: int, archive_name: str) -> None:
    print(f'Persisted cache chunk {done}/{total}: {archive_name}')

cache_manifest = build_cache_chunks(
    all_views, dataset_dir=DATASET_DIR, persistent_dir=PERSISTENT_CACHE_DIR,
    scratch_dir=CACHE_BUILD_SCRATCH, chunk_size=CHUNK_SIZE, workers=WORKERS,
    progress=report_chunk,
)
print(json.dumps(cache_manifest, indent=2))
print('CACHE PREPARATION COMPLETE')


## Next Step

Only after the final code cell prints `CACHE PREPARATION COMPLETE`, open Notebook 14, switch to a T4 GPU, and run all cells. Notebook 14 verifies the view-set digest and archive checksums before training.
